# Ablations and Attribution

Covers everything from the study that does not already have a notebook home: the attention-pooling ablation, the per-attack-type breakdown, the attribution probes (structural and content shortcuts), the computational profile, and holdout attack-type generalization.

All sections here **evaluate already-trained checkpoints** rather than training from scratch. The two full multi-hour trainings this notebook depends on (the main TGN detector, and the attention-pooling ablation variant) are run as standalone background scripts, not notebook cells - see `src/models/train.py` and `src/experiments/train_ablation_meanpool.py`. Only the cheap baselines (Isolation Forest, One-Class SVM, Static GNN) are trained inline, in notebook 05.

In [1]:
import sys
import json
import time
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
from torch.utils.data import DataLoader
from torch_geometric.data import Batch
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

sys.path.insert(0, str(Path("..").resolve()))

from src.data_generation.dataset import MemoryTrajectoryDataset, collate_trajectories
from src.models.tgn_detector import TGNDetector

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

CONFIG_PATH = Path("../configs/default.yaml")
with open(CONFIG_PATH) as f:
    CONFIG = yaml.safe_load(f)

DATA_DIR = Path("../data/raw/llm_generated")
MAIN_RUN_DIR = Path("../experiments/run_current_arch")
print(f"Data directory: {DATA_DIR}")
print(f"Main run directory: {MAIN_RUN_DIR}")

Using device: cuda
Data directory: ..\data\raw\llm_generated
Main run directory: ..\experiments\run_current_arch


## 1. Attention-Pooling Ablation (Mean Pooling)

`TGNDetectorMeanPool` is identical to `TGNDetector` in every respect - node memory GRU, GAT graph encoder, temporal GRU aggregator, graph/node heads - except `AttentionPool` is replaced with plain `global_mean_pool`. This isolates the attention-pooling contribution cleanly (verified below by exact parameter-count matching), unlike the static-graph ablation in the previous notebook, which removes attention pooling, node memory and the temporal aggregator all at once.

This section evaluates the already-trained checkpoint at `experiments/run_ablation_meanpool/phase2_best.pt` (trained by `src/experiments/train_ablation_meanpool.py`, same recipe/seed/data as the main detector - Phase 1 + Phase 2, several hours on a 4GB GPU). It does not retrain.

In [2]:
from typing import List, Tuple, Optional

class TGNDetectorMeanPool(TGNDetector):
    """Ablation variant: AttentionPool -> global_mean_pool. Everything else unchanged."""

    def __init__(self, config: dict):
        super().__init__(config)
        del self.attn_pool  # unused in this variant

    def forward(self, graph_sequence: List[Batch]) -> Tuple[torch.Tensor, torch.Tensor]:
        T = len(graph_sequence)
        graph_embeddings = []
        last_node_embeddings = None
        last_batch_idx = None

        self.reset_memory()

        for t in range(T):
            g = graph_sequence[t]
            x = g.x
            edge_index = g.edge_index
            batch_idx = g.batch
            edge_attr = g.edge_attr if hasattr(g, 'edge_attr') and g.edge_attr is not None else None

            if hasattr(g, 'node_ids') and g.node_ids is not None:
                prev_memory = self._get_node_memories(g.node_ids, x.device)
            else:
                prev_memory = None

            node_mem = self.node_memory(x, prev_memory)

            if hasattr(g, 'node_ids') and g.node_ids is not None:
                self._store_node_memories(g.node_ids, node_mem)

            gat_input = torch.cat([x, node_mem], dim=-1)
            node_embeddings = self.graph_encoder(gat_input, edge_index, edge_attr=edge_attr)
            graph_emb = global_mean_pool(node_embeddings, batch_idx)
            graph_embeddings.append(graph_emb)

            if t == T - 1:
                last_node_embeddings = node_embeddings
                last_batch_idx = batch_idx

        graph_seq_tensor = torch.stack(graph_embeddings, dim=1)
        trajectory_emb = self.temporal_aggregator(graph_seq_tensor)
        graph_scores = self.graph_head(trajectory_emb)
        node_scores = self.node_head(last_node_embeddings, trajectory_emb, last_batch_idx)

        return graph_scores, node_scores

In [3]:
def evaluate_meanpool_ablation(config: dict, checkpoint_path: str, data_dir: str):
    T = config["training"]["temporal_window"]
    embedding_dim = config["simulation"]["embedding_dim"]

    model = TGNDetectorMeanPool(config).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    model.eval()

    full_params = sum(p.numel() for p in TGNDetector(config).parameters())
    meanpool_params = sum(p.numel() for p in model.parameters())
    print(f"Full detector params:   {full_params:,}")
    print(f"Mean-pool variant params: {meanpool_params:,}")
    print(f"Difference: {full_params - meanpool_params:,} (should equal AttentionPool's own param count, 8,321)")

    test_dataset = MemoryTrajectoryDataset(data_dir, split="test", temporal_window=T, embedding_dim=embedding_dim)
    test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=collate_trajectories)

    all_graph_scores, all_graph_labels = [], []
    all_node_scores, all_node_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            graphs = [g.to(device) for g in batch["graphs"]]
            graph_labels = batch["graph_labels"]
            graph_scores, node_scores = model(graphs)

            all_graph_scores.extend(graph_scores.cpu().numpy())
            all_graph_labels.extend(graph_labels.numpy())

            last_graph = graphs[-1]
            if hasattr(last_graph, "node_labels"):
                node_labels = last_graph.node_labels.cpu().numpy()
                ns = node_scores.cpu().numpy()
                min_len = min(len(ns), len(node_labels))
                all_node_scores.extend(ns[:min_len])
                all_node_labels.extend(node_labels[:min_len])

    results = {}
    graph_scores = np.array(all_graph_scores)
    graph_labels = np.array(all_graph_labels)
    if len(np.unique(graph_labels)) > 1:
        results["graph_auroc"] = roc_auc_score(graph_labels, graph_scores)
        results["graph_ap"] = average_precision_score(graph_labels, graph_scores)
        results["graph_f1"] = f1_score(graph_labels, (graph_scores > 0.5).astype(int))

    node_scores = np.array(all_node_scores)
    node_labels = np.array(all_node_labels)
    if len(np.unique(node_labels)) > 1:
        results["node_auroc"] = roc_auc_score(node_labels, node_scores)
        results["node_ap"] = average_precision_score(node_labels, node_scores)

    return results

meanpool_checkpoint = Path("../experiments/run_ablation_meanpool/phase2_best.pt")
meanpool_results = evaluate_meanpool_ablation(CONFIG, str(meanpool_checkpoint), str(DATA_DIR))
print("\n=== Attention-Pooling Ablation (Mean Pooling) Results ===")
for k, v in meanpool_results.items():
    print(f"  {k}: {v:.4f}")

Full detector params:   667,331
Mean-pool variant params: 659,010
Difference: 8,321 (should equal AttentionPool's own param count, 8,321)



=== Attention-Pooling Ablation (Mean Pooling) Results ===
  graph_auroc: 0.9944
  graph_ap: 0.9921
  graph_f1: 0.9576
  node_auroc: 0.9718
  node_ap: 0.6161


## 2. Per-Attack-Type Breakdown

Reports graph-level AUROC/AP separately for each `attack_type` against the shared pool of clean windows, instead of one pooled number, so that no single attack strategy is trivially easy and dragging the average up unnoticed.

Note that the partitioning is not stratified (Section 3.7): the three strategies contribute unequal numbers of poisoned sequences to the test partition (direct-write: 2, the other two: 3 each), which is why the `n` column below does not divide evenly.

In [4]:
def evaluate_attack_type_breakdown(config: dict, checkpoint_path: str, data_dir: str):
    T = config["training"]["temporal_window"]

    with open(Path(data_dir) / "metadata.json") as f:
        metadata = {m["sequence_id"]: m for m in json.load(f)}

    model = TGNDetector(config).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    model.eval()

    test_ds = MemoryTrajectoryDataset(data_dir, split="test", temporal_window=T)
    print(f"Scoring {len(test_ds)} test windows...")

    records = []
    with torch.no_grad():
        for i in range(len(test_ds)):
            item = test_ds[i]
            batch = collate_trajectories([item])
            graphs = [g.to(device) for g in batch["graphs"]]
            graph_scores, _ = model(graphs)
            score = graph_scores.squeeze().item()

            seq_id = item["sequence_id"]
            m = metadata[seq_id]
            attack_type = m["attack_type"] if m["label"] == "poisoned" else "clean"
            records.append((attack_type, item["graph_label"], score))

    by_type = defaultdict(list)
    clean_records = [(y, s) for t, y, s in records if t == "clean"]
    for t, y, s in records:
        by_type[t].append((y, s))

    print("\nPer-attack-type graph-level AUROC/AP (vs. shared pool of clean windows):")
    results = {}
    for attack_type in sorted(t for t in by_type if t != "clean"):
        pool = clean_records + by_type[attack_type]
        y_true = np.array([y for y, s in pool])
        y_score = np.array([s for y, s in pool])
        n_pos = int(y_true.sum())
        auroc = roc_auc_score(y_true, y_score)
        ap = average_precision_score(y_true, y_score)
        results[attack_type] = {
            "auroc": float(auroc), "ap": float(ap),
            "n": int(len(y_true)), "n_positive": n_pos,
            "n_windows_this_type": len(by_type[attack_type]),
        }
        print(f"  {attack_type:20s}: AUROC={auroc:.4f}  AP={ap:.4f}  "
              f"(n={len(y_true)}, pos={n_pos}, windows_of_type={len(by_type[attack_type])})")

    y_true_all = np.array([y for _, y, _ in records])
    y_score_all = np.array([s for _, _, s in records])
    pooled_auroc = roc_auc_score(y_true_all, y_score_all)
    pooled_ap = average_precision_score(y_true_all, y_score_all)
    print(f"\n  {'POOLED (all types)':20s}: AUROC={pooled_auroc:.4f}  AP={pooled_ap:.4f}  (n={len(y_true_all)})")
    results["_pooled"] = {"auroc": float(pooled_auroc), "ap": float(pooled_ap), "n": int(len(y_true_all))}
    return results

attack_type_results = evaluate_attack_type_breakdown(CONFIG, str(MAIN_RUN_DIR / "phase2_best.pt"), str(DATA_DIR))

Scoring 330 test windows...



Per-attack-type graph-level AUROC/AP (vs. shared pool of clean windows):
  direct_write        : AUROC=1.0000  AP=1.0000  (n=264, pos=21, windows_of_type=22)
  incremental_slow    : AUROC=1.0000  AP=1.0000  (n=275, pos=33, windows_of_type=33)
  minja_query_only    : AUROC=1.0000  AP=1.0000  (n=275, pos=32, windows_of_type=33)

  POOLED (all types)  : AUROC=1.0000  AP=1.0000  (n=330)


## 3. Attribution Probes (Structural and Content Shortcuts)

Tests whether the detector's near-perfect performance derives from the mechanism it was designed to exploit, or from a simpler shortcut available in the same data, by restricting a plain logistic regression to progressively richer information and measuring what that restriction alone supports.

- **Structural probe**: `[num_nodes, num_edges, edge_density]` only. No embeddings, no content.
- **Content probe**: mean-pooled node embeddings only. No edges, no temporal order.

The gap between the content probe and the full detector is attributable to relational structure and temporal ordering jointly. The static-graph ablation (notebook 05, trained with node loss added) further decomposes that gap: it has relational structure but no temporal ordering, so comparing it against the content probe isolates structure's share, and comparing the full detector against it isolates temporal ordering's share.

In [5]:
def extract_structural_features(dataset):
    X, y = [], []
    for i in range(len(dataset)):
        item = dataset[i]
        g = item["graphs"][-1]
        n = g.x.size(0)
        e = g.edge_index.size(1)
        density = e / max(n * (n - 1), 1)
        X.append([n, e, density])
        y.append(item["graph_label"])
    return np.array(X, dtype=np.float64), np.array(y, dtype=np.int64)

_EMBEDDING_DIM = 384

def extract_content_features(dataset):
    X, y = [], []
    for i in range(len(dataset)):
        item = dataset[i]
        last_graph = item["graphs"][-1]
        node_embeddings = last_graph.x[:, :_EMBEDDING_DIM].numpy()
        X.append(node_embeddings.mean(axis=0))
        y.append(item["graph_label"])
    return np.stack(X), np.array(y, dtype=np.int64)

T = CONFIG["training"]["temporal_window"]
train_ds = MemoryTrajectoryDataset(DATA_DIR, split="train", temporal_window=T)
test_ds = MemoryTrajectoryDataset(DATA_DIR, split="test", temporal_window=T)

# Structural probe
Xs_train, ys_train = extract_structural_features(train_ds)
Xs_test, ys_test = extract_structural_features(test_ds)
clf_struct = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xs_train, ys_train)
struct_scores = clf_struct.predict_proba(Xs_test)[:, 1]
struct_auroc = roc_auc_score(ys_test, struct_scores)
struct_ap = average_precision_score(ys_test, struct_scores)
print(f"Structural probe: AUROC={struct_auroc:.4f}  AP={struct_ap:.4f}")

# Content probe
Xc_train, yc_train = extract_content_features(train_ds)
Xc_test, yc_test = extract_content_features(test_ds)
clf_content = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xc_train, yc_train)
content_scores = clf_content.predict_proba(Xc_test)[:, 1]
content_auroc = roc_auc_score(yc_test, content_scores)
content_ap = average_precision_score(yc_test, content_scores)
content_f1 = f1_score(yc_test, (content_scores >= 0.5).astype(int))
print(f"Content probe:    AUROC={content_auroc:.4f}  AP={content_ap:.4f}  F1={content_f1:.4f}")

attribution_results = {
    "structural_probe": {"auroc": float(struct_auroc), "ap": float(struct_ap)},
    "content_probe": {"auroc": float(content_auroc), "ap": float(content_ap), "f1": float(content_f1)},
}
print(f"\nGap from content probe to full detector (1.0000): {1.0 - content_auroc:.4f}")
print("Compare against the static-graph ablation's graph_auroc (notebook 05, section 5) "
      "to decompose this gap into structure's share vs temporal ordering's share.")

Structural probe: AUROC=0.5946  AP=0.3888


Content probe:    AUROC=0.7729  AP=0.4388  F1=0.5577

Gap from content probe to full detector (1.0000): 0.2271
Compare against the static-graph ablation's graph_auroc (notebook 05, section 5) to decompose this gap into structure's share vs temporal ordering's share.


## 4. Computational Profile

Parameter count (total + per-submodule), inference latency/throughput (single-window and batched, GPU and CPU), and peak GPU memory footprint for the trained TGN detector.

In [6]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    by_submodule = {}
    for name, module in model.named_children():
        by_submodule[name] = sum(p.numel() for p in module.parameters())
    return {"total": total, "by_submodule": by_submodule}

def time_inference(model, batches, device, n_repeats=30, warmup=5):
    times = []
    with torch.no_grad():
        for i in range(warmup):
            model.reset_memory()
            _ = model(batches[i % len(batches)])
            if device.type == "cuda":
                torch.cuda.synchronize()
        for i in range(n_repeats):
            batch = batches[i % len(batches)]
            model.reset_memory()
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model(batch)
            if device.type == "cuda":
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            times.append(t1 - t0)
    times = np.array(times)
    return {"mean_ms": float(times.mean() * 1000), "p50_ms": float(np.median(times) * 1000),
            "p95_ms": float(np.percentile(times, 95) * 1000)}

checkpoint_path = MAIN_RUN_DIR / "phase2_best.pt"
checkpoint_size_mb = checkpoint_path.stat().st_size / (1024 * 1024)
profile_results = {"checkpoint_size_mb": round(checkpoint_size_mb, 3)}

T = CONFIG["training"]["temporal_window"]
prof_ds = MemoryTrajectoryDataset(DATA_DIR, split="test", temporal_window=T)
n_pool = min(20, len(prof_ds))
items = [prof_ds[i] for i in range(n_pool)]

def make_batches(batch_size, dev):
    out = []
    for start in range(0, len(items) - batch_size + 1, batch_size):
        chunk = items[start:start + batch_size]
        collated = collate_trajectories(chunk)
        graphs = [g.to(dev) for g in collated["graphs"]]
        out.append(graphs)
    return out

model = TGNDetector(CONFIG).to(device)
model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
model.eval()

param_info = count_params(model)
profile_results["parameters"] = param_info
print(f"Total parameters: {param_info['total']:,}")
for name, n in param_info["by_submodule"].items():
    print(f"  {name:20s}: {n:,}")

batches_1 = make_batches(1, device)
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(device)
lat_1 = time_inference(model, batches_1, device)
print(f"\nBatch=1 latency: mean={lat_1['mean_ms']:.2f}ms  p50={lat_1['p50_ms']:.2f}ms  p95={lat_1['p95_ms']:.2f}ms")
profile_results["batch_size_1"] = lat_1
if device.type == "cuda":
    profile_results["peak_gpu_memory_mb"] = round(torch.cuda.max_memory_allocated(device) / (1024 * 1024), 2)
    print(f"Peak GPU memory: {profile_results['peak_gpu_memory_mb']:.1f} MB")

Total parameters: 667,331
  node_memory         : 198,528
  graph_encoder       : 332,416
  attn_pool           : 8,321
  temporal_aggregator : 99,072
  graph_head          : 8,321
  node_head           : 20,673



Batch=1 latency: mean=70.28ms  p50=69.20ms  p95=80.47ms
Peak GPU memory: 56.8 MB


## 5. Holdout Attack-Type Generalization

Evaluates the already-trained holdout checkpoint (trained with `incremental_slow` excluded entirely from train/val via `--holdout-attack-type incremental_slow`) against that excluded attack type, to check whether the detector generalizes to an attack pattern it never trained on, or has only memorized the three fixed attack templates.

This does not retrain - it loads `experiments/run_holdout_incremental_slow/phase2_best.pt`.

In [7]:
sys.path.insert(0, str(Path("..").resolve()))
from src.models.train import Trainer

HOLDOUT_ATTACK_TYPE = "incremental_slow"
HOLDOUT_RUN_DIR = Path("../experiments/run_holdout_incremental_slow")

holdout_trainer = Trainer(CONFIG, str(DATA_DIR), device, holdout_attack_type=HOLDOUT_ATTACK_TYPE)
holdout_state_dict = torch.load(HOLDOUT_RUN_DIR / "phase2_best.pt", map_location=device, weights_only=True)
holdout_trainer.model.load_state_dict(holdout_state_dict)

holdout_result = holdout_trainer.evaluate_holdout()
print(f"\n=== Holdout attack-type generalization ({HOLDOUT_ATTACK_TYPE}) ===")
print(holdout_result)

Model parameters: 667,331
Holdout attack type 'incremental_slow': removed 37 sequences from train/val, 40 sequences reserved for generalization eval.



=== Holdout attack-type generalization (incremental_slow) ===
{'loss': 0.025412839119830593, 'graph_auroc': 0.0, 'node_auroc': 0.8543636617937689, 'n_sequences': 40}


## 6. Save Results

In [8]:
combined_output = {
    "attention_pooling_ablation": meanpool_results,
    "attack_type_breakdown": attack_type_results,
    "attribution_probes": attribution_results,
    "computational_profile": profile_results,
    "holdout_generalization": holdout_result,
}

output_path = Path("../experiments/notebook_07_results.json")
with open(output_path, "w") as f:
    json.dump(combined_output, f, indent=2, default=str)
print(f"Results saved to {output_path}")

Results saved to ..\experiments\notebook_07_results.json
